# LoRA and QLoRA Course Assistant Walkthrough

This notebook mirrors the LoRA/QLoRA homework workflow from the chapter. Use it in Jupyter, Colab, or Kaggle when you want to run the experiment one step at a time. The companion `lora_unsloth_course_assistant.py` script remains the source of truth for the actual baseline, training, metadata, and smoke-check behavior.

Run the syntax and dependency cells anywhere. Run the GPU cells only after setting up a compatible Unsloth environment and verifying the current Unsloth installation instructions.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

In [ ]:
from pathlib import Path
import ast
import json
import os
import platform
import subprocess
import sys


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir("lora_unsloth_course_assistant.py", "chapter_lora_qlora_adaptation")
os.chdir(CHAPTER_DIR)
SCRIPT = CHAPTER_DIR / "lora_unsloth_course_assistant.py"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Chapter directory:", CHAPTER_DIR)
print("Working directory:", Path.cwd())

## Check the Companion Script

This cell verifies that the script is syntactically valid and reports which optional training dependencies are currently available. Missing dependencies are expected on a CPU-only machine.

In [ ]:
ast.parse(SCRIPT.read_text(encoding="utf-8"), filename=str(SCRIPT))
subprocess.run(
    [sys.executable, str(SCRIPT), "--check-deps", "--allow-missing-deps"],
    check=True,
)

## Write Train and Validation JSONL Files

A real run should use more examples than this small teaching set. The important habits are explicit fields, a held-out validation split, and stable file names that the baseline and training commands can reuse.

In [ ]:
import random

records = [
    {
        "question": "How do I improve validation accuracy in an image classifier?",
        "answer": (
            "Start by checking the validation split and baseline. Then change one "
            "factor at a time, such as augmentation, learning rate, model size, "
            "or input resolution. Keep the final test set untouched."
        ),
    },
    {
        "question": "Why should I not tune hyperparameters on the test set?",
        "answer": (
            "The test set estimates performance after model selection. If it is "
            "used repeatedly during tuning, it becomes another validation set and "
            "the final number is no longer an honest estimate."
        ),
    },
    {
        "question": "When is LoRA a better first step than full fine-tuning?",
        "answer": (
            "LoRA is a good first step when the base model is already useful and "
            "you need a cheaper adaptation of style, format, or domain behavior. "
            "It trains far fewer parameters and uses less optimizer memory."
        ),
    },
    {
        "question": "What should I record in a supervised fine-tuning run?",
        "answer": (
            "Record the checkpoint, tokenizer, data split, context length, batch "
            "size, gradient accumulation, learning rate, seed, package versions, "
            "hardware, runtime, memory, and validation results."
        ),
    },
    {
        "question": "How do I diagnose overfitting in a small adaptation run?",
        "answer": (
            "Compare training behavior with held-out validation prompts. If the "
            "adapter memorizes the training examples but validation quality stalls "
            "or degrades, reduce capacity, data leakage, or training steps."
        ),
    },
    {
        "question": "Why does the prompt-only baseline matter?",
        "answer": (
            "It shows what the selected base model already does with the same "
            "prompt format, decoding settings, and validation prompts. The adapter "
            "has to beat that control to justify the extra complexity."
        ),
    },
    {
        "question": "What does the LoRA rank control?",
        "answer": (
            "The rank controls the size of the low-rank update matrices. Higher "
            "rank usually adds more trainable parameters and capacity, but it can "
            "also increase memory use and overfitting risk."
        ),
    },
    {
        "question": "Why can quantization reduce fine-tuning memory?",
        "answer": (
            "Quantization stores the frozen base weights with fewer bits. In QLoRA, "
            "the quantized base remains mostly frozen while small LoRA adapters "
            "carry the trainable update."
        ),
    },
    {
        "question": "What makes an ablation controlled?",
        "answer": (
            "A controlled ablation changes one factor while holding the model, data "
            "split, decoding, and evaluation method fixed. That makes the comparison "
            "easier to interpret."
        ),
    },
    {
        "question": "How should I choose a learning rate for a small LoRA run?",
        "answer": (
            "Start from a conservative value recommended for the chosen workflow, "
            "watch validation behavior, and change only one setting at a time. "
            "Do not choose the rate by repeatedly checking the final test set."
        ),
    },
    {
        "question": "Why save adapter metadata with the model artifact?",
        "answer": (
            "The adapter alone is not enough to reproduce the run. Metadata ties "
            "the artifact to the base checkpoint, tokenizer, LoRA settings, data, "
            "software versions, hardware, and evaluation method."
        ),
    },
    {
        "question": "What should I do if the adapter output is worse than baseline?",
        "answer": (
            "Keep the baseline result, inspect failure examples, and treat the run "
            "as evidence. Try a controlled change such as data cleanup, fewer steps, "
            "a different rank, or a revised prompt template."
        ),
    },
]

rng = random.Random(3407)
rng.shuffle(records)
validation_records = records[:5]
training_records = records[5:]

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, sort_keys=True) + "\n")

write_jsonl("validation.jsonl", validation_records)
write_jsonl("train.jsonl", training_records)
print(f"Wrote {len(training_records)} training and {len(validation_records)} validation records.")

### Reader Checkpoint

After the previous cell runs, confirm that the printed paths, shapes, commands, or tables match the section description before moving on. If this checkpoint fails in a public-repo environment, fix dependencies, data paths, or guarded flags before starting longer runs.

## Preview the Prompt Format

The default script command prints a small built-in preview and reminds you to choose either the baseline or training path for a real run.

In [ ]:
subprocess.run([sys.executable, str(SCRIPT)], check=True)

## Run the Prompt-Only Baseline

Set `RUN_BASELINE` to `True` only in a compatible Unsloth GPU environment. The baseline uses the same validation prompts and decoding settings without attaching adapters.

In [ ]:
RUN_BASELINE = False

if RUN_BASELINE:
    subprocess.run(
        [
            sys.executable,
            str(SCRIPT),
            "--baseline",
            "--validation-data",
            "validation.jsonl",
            "--baseline-output",
            "runs/unsloth_lora/baseline_outputs.jsonl",
            "--output-dir",
            "runs/unsloth_lora",
            "--run-name",
            "course_assistant",
        ],
        check=True,
    )
else:
    print("Skipped baseline. Set RUN_BASELINE = True in a compatible Unsloth GPU runtime.")

## Run LoRA or QLoRA Fine-Tuning

Set `RUN_TRAINING` to `True` only after the baseline works. Keep the checkpoint, data split, context length, LoRA settings, runtime, memory, and validation method in the final report.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(
        [
            sys.executable,
            str(SCRIPT),
            "--train",
            "--data",
            "train.jsonl",
            "--run-kind",
            "lora",
            "--output-dir",
            "runs/unsloth_lora",
            "--run-name",
            "course_assistant",
            "--max-seq-length",
            "2048",
            "--lora-rank",
            "16",
            "--lora-alpha",
            "16",
            "--learning-rate",
            "2e-4",
            "--max-steps",
            "60",
        ],
        check=True,
    )
else:
    print("Skipped training. Set RUN_TRAINING = True in a compatible Unsloth GPU runtime.")

## Run One Controlled Ablation

This example changes only the LoRA rank. You can instead ablate sequence length, learning rate, number of examples, target modules, or quantized base loading.

In [ ]:
RUN_ABLATION = False

if RUN_ABLATION:
    subprocess.run(
        [
            sys.executable,
            str(SCRIPT),
            "--train",
            "--data",
            "train.jsonl",
            "--run-kind",
            "ablation",
            "--run-name",
            "rank_8",
            "--output-dir",
            "runs/unsloth_lora_rank_8",
            "--lora-rank",
            "8",
        ],
        check=True,
    )
else:
    print("Skipped ablation. Set RUN_ABLATION = True after choosing one controlled change.")

## Inspect Run Metadata

After running the baseline or training cells, inspect the metadata files before writing the report. The metadata should make clear which run is baseline, LoRA/QLoRA, or ablation.

In [ ]:
metadata_paths = sorted(Path("runs").glob("**/*metadata.json"))
summary_keys = [
    "run_kind",
    "run_name",
    "model_name",
    "training_records",
    "records",
    "max_seq_length",
    "load_in_4bit",
    "lora_rank",
    "lora_alpha",
    "learning_rate",
    "trainable_parameters",
    "wall_clock_seconds",
]

if not metadata_paths:
    print("No run metadata found yet.")

for path in metadata_paths:
    payload = json.loads(path.read_text(encoding="utf-8"))
    summary = {key: payload.get(key) for key in summary_keys if key in payload}
    print(path)
    print(json.dumps(summary, indent=2, sort_keys=True))

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.